# PM2.5/PM10 Training for Colab (Firebase + Node-RED)

Workflow:
1. Load hourly data from Firebase RTDB REST (or local fallback CSV).
2. Clean data and drop today and yesterday from the training subset.
3. Build daily features, train PM2.5/PM10 models, and evaluate.
4. Forecast next day from latest available history (to avoid stale forecast dates).
5. Save artifacts for Node-RED dashboard running on Raspberry Pi 4.

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from datetime import datetime
from pathlib import Path
from zoneinfo import ZoneInfo

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from sklearn.ensemble import RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline

warnings.filterwarnings("ignore")

In [ ]:
# -----------------------------
# Configuration
# -----------------------------
TIMEZONE = os.getenv("TIMEZONE", "Asia/Bangkok")
BANGKOK_TZ = ZoneInfo(TIMEZONE)
RANDOM_STATE = int(os.getenv("RANDOM_STATE", "42"))


def _env_bool(name: str, default: bool = False) -> bool:
    raw = os.getenv(name)
    if raw is None:
        return default
    return str(raw).strip().lower() in {"1", "true", "yes", "y", "on"}


def now_bangkok() -> datetime:
    return datetime.now(BANGKOK_TZ)


IS_COLAB = "COLAB_RELEASE_TAG" in os.environ

# Firebase RTDB REST config
FIREBASE_RTD_URL = os.getenv("FIREBASE_RTD_URL", "").strip()
FIREBASE_PATH = os.getenv("FIREBASE_PATH", "/AirQuality/Log").strip() or "/AirQuality/Log"
FIREBASE_AUTH = os.getenv("FIREBASE_AUTH", "").strip()

# If your ESP32 path changes or contains random push IDs, the loader will try these paths in order.
_DEFAULT_PATH_CANDIDATES = [
    FIREBASE_PATH,
    "/AirQuality/Log",
    "/air_quality_readings",
    "/AirQuality",
    "/",
]
path_candidates_env = os.getenv("FIREBASE_PATH_CANDIDATES", "").strip()
if path_candidates_env:
    FIREBASE_PATH_CANDIDATES = [p.strip() for p in path_candidates_env.split(",") if p.strip()]
else:
    FIREBASE_PATH_CANDIDATES = _DEFAULT_PATH_CANDIDATES

# Local historical fallback for Colab (upload this file in Colab Files panel if needed)
default_local_fallback = "/content/openmeteo_bangkok_hourly_2023_2026.csv" if IS_COLAB else "data/openmeteo_bangkok_hourly_2023_2026.csv"
USE_LOCAL_FALLBACK = _env_bool("USE_LOCAL_FALLBACK", True)
LOCAL_FALLBACK_PATH = os.getenv("LOCAL_FALLBACK_PATH", default_local_fallback).strip()

# Mix local history + latest Firebase rows (recommended when Firebase has only recent dates)
MERGE_LOCAL_WITH_FIREBASE = _env_bool("MERGE_LOCAL_WITH_FIREBASE", True)

# Output directory
default_models_dir = "/content/models" if IS_COLAB else "models"
MODELS_DIR = Path(os.getenv("MODELS_DIR", default_models_dir).strip())
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print("MODELS_DIR:", MODELS_DIR)
print("FIREBASE_PATH_CANDIDATES:", FIREBASE_PATH_CANDIDATES)


In [ ]:
def _build_endpoint(base_url: str, path: str) -> str:
    endpoint = f"{base_url.rstrip('/')}/{str(path).lstrip('/')}"
    if not endpoint.endswith(".json"):
        endpoint = endpoint.rstrip("/") + ".json"
    return endpoint


def _firebase_get(base_url: str, path: str, auth: str = ""):
    endpoint = _build_endpoint(base_url, path)
    params = {}
    if auth.strip():
        params["auth"] = auth.strip()

    resp = requests.get(endpoint, params=params, timeout=60)
    resp.raise_for_status()
    return resp.json()


def _looks_like_measurement_record(obj: dict) -> bool:
    keys = {str(k).strip().lower() for k in obj.keys()}
    has_pm = any(k in keys for k in ["pm25", "pm2_5", "pm2.5", "pm10"])
    has_time = any(k in keys for k in ["timestamp", "time", "ts", "ts_device", "created_at", "datetime"])
    return has_pm and has_time


def _collect_measurement_records(payload, parent_path: str = "") -> list[dict]:
    records: list[dict] = []

    if isinstance(payload, dict):
        if _looks_like_measurement_record(payload):
            rec = {"_path": parent_path}
            rec.update(payload)
            records.append(rec)

        for k, v in payload.items():
            child_path = f"{parent_path}/{k}" if parent_path else str(k)
            records.extend(_collect_measurement_records(v, child_path))

    elif isinstance(payload, list):
        for i, item in enumerate(payload):
            child_path = f"{parent_path}/{i}" if parent_path else str(i)
            records.extend(_collect_measurement_records(item, child_path))

    return records


def fetch_firebase_rtdb(url: str, path: str, auth: str = "") -> pd.DataFrame:
    if not url.strip():
        return pd.DataFrame()

    payload = _firebase_get(url, path, auth)
    if payload is None:
        return pd.DataFrame()

    records = _collect_measurement_records(payload, parent_path=path.strip("/") or "/")
    if records:
        return pd.DataFrame(records)

    # Fallbacks for flat structures
    if isinstance(payload, dict):
        if all(isinstance(v, dict) for v in payload.values()):
            rows = [{"_key": k, "_path": f"{path.rstrip('/')}/{k}", **v} for k, v in payload.items()]
            return pd.DataFrame(rows)
        return pd.DataFrame([payload])

    if isinstance(payload, list):
        return pd.DataFrame(payload)

    return pd.DataFrame()


def pick_first_existing(df: pd.DataFrame, candidates: list[str]) -> str | None:
    for c in candidates:
        if c in df.columns:
            return c
    return None


def fetch_firebase_from_candidates(url: str, candidates: list[str], auth: str = "") -> tuple[pd.DataFrame, str | None, list[str]]:
    tried_paths: list[str] = []

    for p in candidates:
        if not str(p).strip():
            continue

        norm = "/" + str(p).strip().lstrip("/")
        tried_paths.append(norm)

        try:
            df = fetch_firebase_rtdb(url, norm, auth)
        except Exception as exc:
            print(f"Path failed: {norm} -> {exc}")
            continue

        if not df.empty:
            return df, norm, tried_paths

    return pd.DataFrame(), None, tried_paths


firebase_df = pd.DataFrame()
local_df = pd.DataFrame()
raw_df = pd.DataFrame()
used_path = None
tried: list[str] = []

if FIREBASE_RTD_URL.strip():
    firebase_df, used_path, tried = fetch_firebase_from_candidates(
        FIREBASE_RTD_URL,
        FIREBASE_PATH_CANDIDATES,
        FIREBASE_AUTH,
    )

    if used_path:
        firebase_df = firebase_df.copy()
        firebase_df["_source"] = "firebase"
        print(f"Loaded Firebase rows: {len(firebase_df):,} (path: {used_path})")
        if "_path" in firebase_df.columns and firebase_df["_path"].notna().any():
            print("Example source path:", firebase_df["_path"].dropna().iloc[0])
    else:
        print("No rows from Firebase. Tried paths:", tried)
else:
    print("Firebase URL is empty. Skip Firebase fetch.")

if USE_LOCAL_FALLBACK:
    try:
        local_df = pd.read_csv(LOCAL_FALLBACK_PATH)
        local_df = local_df.copy()
        local_df["_source"] = "local_fallback"
        print(f"Loaded local fallback rows: {len(local_df):,} ({LOCAL_FALLBACK_PATH})")
    except Exception as exc:
        print(f"Local fallback load failed: {LOCAL_FALLBACK_PATH} -> {exc}")

if MERGE_LOCAL_WITH_FIREBASE and (not local_df.empty) and (not firebase_df.empty):
    raw_df = pd.concat([local_df, firebase_df], ignore_index=True, sort=False)
    print(f"Merged source rows (local + firebase): {len(raw_df):,}")
elif not firebase_df.empty:
    raw_df = firebase_df
elif not local_df.empty:
    raw_df = local_df

if raw_df.empty:
    raise ValueError("No source data found from Firebase and local fallback. Check FIREBASE_PATH_CANDIDATES, LOCAL_FALLBACK_PATH, and Firebase rules.")

raw_df.head()

In [ ]:
def parse_timestamp_series(ts: pd.Series, timezone_name: str = TIMEZONE) -> pd.Series:
    raw = ts.copy()
    as_num = pd.to_numeric(raw, errors="coerce")
    parsed = pd.Series(pd.NaT, index=raw.index, dtype="datetime64[ns]")

    num_mask = as_num.notna()
    if num_mask.any():
        max_abs = float(as_num[num_mask].abs().max())
        if max_abs > 1e14:
            unit = "ns"
        elif max_abs > 1e11:
            unit = "ms"
        else:
            unit = "s"

        parsed_num = pd.to_datetime(as_num[num_mask], unit=unit, errors="coerce", utc=True)
        parsed.loc[num_mask] = parsed_num.dt.tz_convert(timezone_name).dt.tz_localize(None).values

    str_mask = ~num_mask
    if str_mask.any():
        raw_str = raw[str_mask].astype(str).str.strip()
        has_tz_hint = raw_str.str.contains(r"(Z|[+-]\d{2}:?\d{2})$", regex=True, na=False)

        if has_tz_hint.any():
            parsed_with_tz = pd.to_datetime(raw_str[has_tz_hint], errors="coerce", utc=True)
            parsed.loc[raw_str[has_tz_hint].index] = (
                parsed_with_tz.dt.tz_convert(timezone_name).dt.tz_localize(None).values
            )

        no_tz_mask = ~has_tz_hint
        if no_tz_mask.any():
            # No timezone hint means local wall-clock time from device.
            parsed_local = pd.to_datetime(raw_str[no_tz_mask], errors="coerce")
            parsed.loc[raw_str[no_tz_mask].index] = parsed_local.values

    return parsed


def _coalesce_numeric_columns(df: pd.DataFrame, candidates: list[str]) -> pd.Series:
    out = pd.Series(np.nan, index=df.index, dtype="float64")
    for c in candidates:
        if c in df.columns:
            out = out.fillna(pd.to_numeric(df[c], errors="coerce"))
    return out


def _coalesce_timestamp_columns(df: pd.DataFrame, candidates: list[str], timezone_name: str) -> pd.Series:
    out = pd.Series(pd.NaT, index=df.index, dtype="datetime64[ns]")
    for c in candidates:
        if c in df.columns:
            out = out.fillna(parse_timestamp_series(df[c], timezone_name=timezone_name))
    return out


def prepare_hourly_df(df: pd.DataFrame, timezone_name: str = TIMEZONE) -> pd.DataFrame:
    out = df.copy()
    out.columns = [str(c).strip() for c in out.columns]

    time_candidates = ["time", "timestamp", "ts", "ts_device", "created_at", "datetime"]
    out["time"] = _coalesce_timestamp_columns(out, time_candidates, timezone_name=timezone_name)
    if out["time"].notna().sum() == 0:
        raise ValueError("No timestamp column found. Expected one of: time, timestamp, ts, ts_device, created_at, datetime")

    mapping = {
        "pm25": ["pm25", "pm2_5", "pm2.5", "pm25_value"],
        "pm10": ["pm10", "pm10_value"],
        "temperature": ["temperature", "temp", "temperature_c"],
        "humidity": ["humidity", "rh", "humidity_pct"],
    }

    for dst, candidates in mapping.items():
        out[dst] = _coalesce_numeric_columns(out, candidates)

    out = out.dropna(subset=["time"]).copy()

    if "_source" in out.columns:
        source_priority = {"local_fallback": 0, "firebase": 1}
        out["_source_priority"] = out["_source"].map(source_priority).fillna(0)
        out = out.sort_values(["time", "_source_priority"]).drop_duplicates(subset=["time"], keep="last")
        out = out.drop(columns=["_source_priority"])
    else:
        out = out.sort_values("time").drop_duplicates(subset=["time"], keep="last")

    out = out[(out["pm25"].between(0, 500)) | (out["pm25"].isna())]
    out = out[(out["pm10"].between(0, 600)) | (out["pm10"].isna())]

    out = out.sort_values("time").reset_index(drop=True)
    keep_cols = ["time", "pm25", "pm10", "temperature", "humidity"]
    if "_source" in out.columns:
        keep_cols.append("_source")
    return out[keep_cols]


hourly_df = prepare_hourly_df(raw_df)
print("Hourly rows after cleaning:", len(hourly_df))
if "_source" in hourly_df.columns:
    print("Rows by source:")
    display(hourly_df["_source"].value_counts(dropna=False).to_frame("rows"))
display(hourly_df.head())
display(hourly_df.tail())

In [ ]:
# Daily aggregation from source
daily_df = (
    hourly_df.assign(date=hourly_df["time"].dt.normalize())
    .groupby("date", as_index=False)
    .agg(
        pm25=("pm25", "mean"),
        pm10=("pm10", "mean"),
        temperature=("temperature", "mean"),
        humidity=("humidity", "mean"),
        active_samples=("time", "count"),
    )
    .sort_values("date")
    .reset_index(drop=True)
)

today_bkk = pd.Timestamp.now(tz=TIMEZONE).floor("D").tz_localize(None)
cutoff_train_date = today_bkk - pd.Timedelta(days=2)

train_daily = daily_df[daily_df["date"] <= cutoff_train_date].copy()
recent_daily = daily_df[daily_df["date"] > cutoff_train_date].copy()

print("Daily rows total:", len(daily_df))
print("Training cutoff date:", cutoff_train_date.date())
print("Training rows after dropping today+yesterday window:", len(train_daily))
print("Recent rows excluded from training:", len(recent_daily))

display(train_daily.tail())
display(recent_daily.head())

In [ ]:
def download_openmeteo_daily(start_date: str, end_date: str) -> pd.DataFrame:
    url = "https://air-quality-api.open-meteo.com/v1/air-quality"
    params = {
        "latitude": 13.7563,
        "longitude": 100.5018,
        "timezone": TIMEZONE,
        "hourly": "pm2_5,pm10,nitrogen_dioxide,ozone",
        "start_date": start_date,
        "end_date": end_date,
    }
    resp = requests.get(url, params=params, timeout=60)
    resp.raise_for_status()
    payload = resp.json().get("hourly", {})

    hourly = pd.DataFrame({
        "time": payload.get("time", []),
        "om_pm2_5": payload.get("pm2_5", []),
        "om_pm10": payload.get("pm10", []),
        "om_no2": payload.get("nitrogen_dioxide", []),
        "om_o3": payload.get("ozone", []),
    })

    hourly["time"] = pd.to_datetime(hourly["time"], errors="coerce")
    hourly = hourly.dropna(subset=["time"]).copy()
    hourly["date"] = hourly["time"].dt.normalize()

    daily = (
        hourly.groupby("date", as_index=False)
        .mean(numeric_only=True)
        .sort_values("date")
        .reset_index(drop=True)
    )
    return daily


if len(daily_df) == 0:
    raise ValueError("No daily rows available for training.")

start_date = str(daily_df["date"].min().date())
end_date = str(daily_df["date"].max().date())
openmeteo_daily = download_openmeteo_daily(start_date=start_date, end_date=end_date)

print("Open-Meteo daily rows:", len(openmeteo_daily))
display(openmeteo_daily.head())

In [ ]:
def build_feature_table(base_daily: pd.DataFrame, om_daily: pd.DataFrame) -> tuple[pd.DataFrame, list[str], list[str]]:
    merged = (
        base_daily.merge(om_daily, on="date", how="left")
        .sort_values("date")
        .reset_index(drop=True)
    )

    merged["pm25_target"] = merged["pm25"]
    merged["pm10_target"] = merged["pm10"]

    merged["day_of_week"] = merged["date"].dt.dayofweek
    merged["month"] = merged["date"].dt.month
    merged["day_of_year"] = merged["date"].dt.dayofyear
    merged["week_of_year"] = merged["date"].dt.isocalendar().week.astype(int)
    merged["is_weekend"] = merged["day_of_week"].isin([5, 6]).astype(int)

    merged["month_sin"] = np.sin(2 * np.pi * merged["month"] / 12)
    merged["month_cos"] = np.cos(2 * np.pi * merged["month"] / 12)
    merged["dow_sin"] = np.sin(2 * np.pi * merged["day_of_week"] / 7)
    merged["dow_cos"] = np.cos(2 * np.pi * merged["day_of_week"] / 7)

    signal_cols = [
        "pm25_target",
        "pm10_target",
        "temperature",
        "humidity",
        "active_samples",
        "om_pm2_5",
        "om_pm10",
        "om_no2",
        "om_o3",
    ]

    for col in signal_cols:
        if col not in merged.columns:
            merged[col] = np.nan

    feature_cols = [
        "day_of_week",
        "month",
        "day_of_year",
        "week_of_year",
        "is_weekend",
        "month_sin",
        "month_cos",
        "dow_sin",
        "dow_cos",
    ]

    for col in signal_cols:
        lag1_col = f"{col}_lag1"
        lag7_col = f"{col}_lag7"
        roll7_col = f"{col}_roll7"

        merged[lag1_col] = merged[col].shift(1)
        merged[lag7_col] = merged[col].shift(7)
        merged[roll7_col] = merged[col].shift(1).rolling(window=7, min_periods=2).mean()

        feature_cols.extend([lag1_col, lag7_col, roll7_col])

    return merged, feature_cols, signal_cols


merged_daily, feature_cols, signal_cols = build_feature_table(daily_df, openmeteo_daily)

model_df = merged_daily[merged_daily["date"] <= cutoff_train_date].copy()
model_df = model_df.dropna(subset=["pm25_target", "pm10_target"], how="all")
model_df = model_df.sort_values("date").reset_index(drop=True)

print("Merged rows (all history):", len(merged_daily))
print("Model rows (train subset):", len(model_df))
print("Feature count:", len(feature_cols))
display(model_df[["date", "pm25_target", "pm10_target"]].tail())

In [ ]:
min_rows_for_split = 20
if len(model_df) < min_rows_for_split:
    print(
        f"Warning: only {len(model_df)} daily rows after filtering; "
        "use lightweight training/evaluation mode."
    )

X = model_df[feature_cols].copy()
y_pm25 = model_df["pm25_target"].copy()
y_pm10 = model_df["pm10_target"].copy()


def build_regression_pipeline(n_samples: int) -> Pipeline:
    if n_samples < 300:
        base_model = Ridge(alpha=1.0)
    else:
        base_model = RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", base_model),
    ])


def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
    }


results: list[dict] = []
plot_dates = None
plot_actual = None
plot_pred = None

if len(model_df) >= min_rows_for_split:
    X_train_val, X_test, y25_train_val, y25_test = train_test_split(
        X, y_pm25, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )
    _, _, y10_train_val, y10_test = train_test_split(
        X, y_pm10, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )

    X_train, X_val, y25_train, y25_val = train_test_split(
        X_train_val, y25_train_val, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )
    _, _, y10_train, y10_val = train_test_split(
        X_train_val, y10_train_val, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )

    pm25_model = build_regression_pipeline(len(X_train))
    pm10_model = build_regression_pipeline(len(X_train))
    pm25_model.fit(X_train, y25_train)
    pm10_model.fit(X_train, y10_train)

    for split_name, X_part, y25_part, y10_part in [
        ("validation", X_val, y25_val, y10_val),
        ("test", X_test, y25_test, y10_test),
    ]:
        results.append({"target": "pm25", "split": split_name, **regression_metrics(y25_part, pm25_model.predict(X_part))})
        results.append({"target": "pm10", "split": split_name, **regression_metrics(y10_part, pm10_model.predict(X_part))})

    plot_dates = model_df["date"].iloc[-len(X_test):]
    plot_actual = y25_test.values
    plot_pred = pm25_model.predict(X_test)

elif len(model_df) >= 3:
    X_train, X_test, y25_train, y25_test = train_test_split(
        X, y_pm25, test_size=0.33, shuffle=False, random_state=RANDOM_STATE
    )
    _, _, y10_train, y10_test = train_test_split(
        X, y_pm10, test_size=0.33, shuffle=False, random_state=RANDOM_STATE
    )

    pm25_model = build_regression_pipeline(len(X_train))
    pm10_model = build_regression_pipeline(len(X_train))
    pm25_model.fit(X_train, y25_train)
    pm10_model.fit(X_train, y10_train)

    results.append({"target": "pm25", "split": "test_small", **regression_metrics(y25_test, pm25_model.predict(X_test))})
    results.append({"target": "pm10", "split": "test_small", **regression_metrics(y10_test, pm10_model.predict(X_test))})

    plot_dates = model_df["date"].iloc[-len(X_test):]
    plot_actual = y25_test.values
    plot_pred = pm25_model.predict(X_test)

elif len(model_df) >= 1:
    pm25_model = build_regression_pipeline(len(X))
    pm10_model = build_regression_pipeline(len(X))
    pm25_model.fit(X, y_pm25)
    pm10_model.fit(X, y_pm10)

    # Train-only metrics are for sanity checks when history is too short.
    results.append({"target": "pm25", "split": "train_only", **regression_metrics(y_pm25, pm25_model.predict(X))})
    results.append({"target": "pm10", "split": "train_only", **regression_metrics(y_pm10, pm10_model.predict(X))})

else:
    raise ValueError("No model rows available after preprocessing.")

metrics_df = pd.DataFrame(results)
display(metrics_df)

if plot_dates is not None and len(plot_dates) > 0:
    plt.figure(figsize=(10, 4))
    plt.plot(plot_dates, plot_actual, label="PM2.5 actual")
    plt.plot(plot_dates, plot_pred, label="PM2.5 predicted")
    plt.title("PM2.5: actual vs predicted")
    plt.xticks(rotation=30)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
min_rows_for_split = 20
if len(model_df) < min_rows_for_split:
    print(
        f"Warning: only {len(model_df)} daily rows after filtering; "
        "use lightweight training/evaluation mode."
    )

X = model_df[feature_cols].copy()
y_pm25 = model_df["pm25_target"].copy()
y_pm10 = model_df["pm10_target"].copy()


def build_regression_pipeline(n_samples: int) -> Pipeline:
    if n_samples < 300:
        base_model = Ridge(alpha=1.0)
    else:
        base_model = RandomForestRegressor(
            n_estimators=500,
            min_samples_leaf=2,
            n_jobs=-1,
            random_state=RANDOM_STATE,
        )
    return Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("model", base_model),
    ])


def regression_metrics(y_true: pd.Series, y_pred: np.ndarray) -> dict:
    return {
        "MAE": float(mean_absolute_error(y_true, y_pred)),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
        "R2": float(r2_score(y_true, y_pred)) if len(y_true) > 1 else np.nan,
    }


results: list[dict] = []
plot_dates = None
plot_actual = None
plot_pred = None

if len(model_df) >= min_rows_for_split:
    X_train_val, X_test, y25_train_val, y25_test = train_test_split(
        X, y_pm25, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )
    _, _, y10_train_val, y10_test = train_test_split(
        X, y_pm10, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )

    X_train, X_val, y25_train, y25_val = train_test_split(
        X_train_val, y25_train_val, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )
    _, _, y10_train, y10_val = train_test_split(
        X_train_val, y10_train_val, test_size=0.15, shuffle=False, random_state=RANDOM_STATE
    )

    pm25_model = build_regression_pipeline(len(X_train))
    pm10_model = build_regression_pipeline(len(X_train))
    pm25_model.fit(X_train, y25_train)
    pm10_model.fit(X_train, y10_train)

    for split_name, X_part, y25_part, y10_part in [
        ("validation", X_val, y25_val, y10_val),
        ("test", X_test, y25_test, y10_test),
    ]:
        results.append({"target": "pm25", "split": split_name, **regression_metrics(y25_part, pm25_model.predict(X_part))})
        results.append({"target": "pm10", "split": split_name, **regression_metrics(y10_part, pm10_model.predict(X_part))})

    plot_dates = model_df["date"].iloc[-len(X_test):]
    plot_actual = y25_test.values
    plot_pred = pm25_model.predict(X_test)

elif len(model_df) >= 3:
    X_train, X_test, y25_train, y25_test = train_test_split(
        X, y_pm25, test_size=0.33, shuffle=False, random_state=RANDOM_STATE
    )
    _, _, y10_train, y10_test = train_test_split(
        X, y_pm10, test_size=0.33, shuffle=False, random_state=RANDOM_STATE
    )

    pm25_model = build_regression_pipeline(len(X_train))
    pm10_model = build_regression_pipeline(len(X_train))
    pm25_model.fit(X_train, y25_train)
    pm10_model.fit(X_train, y10_train)

    results.append({"target": "pm25", "split": "test_small", **regression_metrics(y25_test, pm25_model.predict(X_test))})
    results.append({"target": "pm10", "split": "test_small", **regression_metrics(y10_test, pm10_model.predict(X_test))})

    plot_dates = model_df["date"].iloc[-len(X_test):]
    plot_actual = y25_test.values
    plot_pred = pm25_model.predict(X_test)

elif len(model_df) >= 1:
    pm25_model = build_regression_pipeline(len(X))
    pm10_model = build_regression_pipeline(len(X))
    pm25_model.fit(X, y_pm25)
    pm10_model.fit(X, y_pm10)

    # Train-only metrics are for sanity checks when history is too short.
    results.append({"target": "pm25", "split": "train_only", **regression_metrics(y_pm25, pm25_model.predict(X))})
    results.append({"target": "pm10", "split": "train_only", **regression_metrics(y_pm10, pm10_model.predict(X))})

else:
    raise ValueError("No model rows available after preprocessing.")

metrics_df = pd.DataFrame(results)
display(metrics_df)

if plot_dates is not None and len(plot_dates) > 0:
    plt.figure(figsize=(10, 4))
    plt.plot(plot_dates, plot_actual, label="PM2.5 actual")
    plt.plot(plot_dates, plot_pred, label="PM2.5 predicted")
    plt.title("PM2.5: actual vs predicted")
    plt.xticks(rotation=30)
    plt.legend()
    plt.tight_layout()
    plt.show()

# Save artifacts
pm25_model_path = MODELS_DIR / "pm25_model.joblib"
pm10_model_path = MODELS_DIR / "pm10_model.joblib"
meta_path = MODELS_DIR / "training_metadata_colab_firebase.json"
forecast_path = MODELS_DIR / "forecast_next_day.json"

joblib.dump(pm25_model, pm25_model_path)
joblib.dump(pm10_model, pm10_model_path)

metadata = {
    "created_at": now_bangkok().isoformat(timespec="seconds"),
    "timezone": TIMEZONE,
    "training_cutoff_date": str(cutoff_train_date.date()),
    "feature_columns": feature_cols,
    "rows_daily_total": int(len(daily_df)),
    "rows_model_subset": int(len(model_df)),
    "metrics": metrics_df.to_dict(orient="records"),
}

with open(meta_path, "w", encoding="utf-8") as f:
    json.dump(metadata, f, ensure_ascii=False, indent=2)

forecast_payload = {
    "generated_at": now_bangkok().isoformat(timespec="seconds"),
    "latest_history_date": str(latest_history_date.date()),
    "forecast_date": str(next_date.date()),
    "pm25_forecast": round(next_pm25, 2),
    "pm10_forecast": round(next_pm10, 2),
    "is_stale_vs_today": is_stale_vs_today,
}

with open(forecast_path, "w", encoding="utf-8") as f:
    json.dump(forecast_payload, f, ensure_ascii=False, indent=2)

print("saved:", pm25_model_path)
print("saved:", pm10_model_path)
print("saved:", meta_path)
print("saved:", forecast_path)


In [ ]:
# Optional: push forecast result back to Firebase for Node-RED dashboard
PUSH_FORECAST_TO_FIREBASE = _env_bool("PUSH_FORECAST_TO_FIREBASE", True)
FIREBASE_FORECAST_PATH = os.getenv("FIREBASE_FORECAST_PATH", "/predictions/next_day.json")
FIREBASE_PREVIOUS_DAY_PATH = os.getenv("FIREBASE_PREVIOUS_DAY_PATH", "/predictions/previous_day.json")
FIREBASE_HISTORY_BASE_PATH = os.getenv("FIREBASE_HISTORY_BASE_PATH", "/predictions/history")


def _build_endpoint(base_url: str, path: str) -> str:
    endpoint = f"{base_url.rstrip('/')}/{path.lstrip('/')}"
    if not endpoint.endswith(".json"):
        endpoint = endpoint.rstrip("/") + ".json"
    return endpoint


def _firebase_request(method: str, path: str, payload: dict | None = None):
    endpoint = _build_endpoint(FIREBASE_RTD_URL, path)

    params = {}
    if FIREBASE_AUTH.strip():
        params["auth"] = FIREBASE_AUTH.strip()

    if method == "GET":
        resp = requests.get(endpoint, params=params, timeout=60)
    elif method == "PUT":
        resp = requests.put(endpoint, params=params, json=payload, timeout=60)
    elif method == "PATCH":
        resp = requests.patch(endpoint, params=params, json=payload, timeout=60)
    else:
        raise ValueError(f"Unsupported method: {method}")

    resp.raise_for_status()
    if resp.content:
        return resp.json()
    return None


if "forecast_payload" not in globals():
    raise ValueError("forecast_payload not found. Run Cell 10 first.")

if PUSH_FORECAST_TO_FIREBASE and FIREBASE_RTD_URL.strip():
    try:
        old_next = _firebase_request("GET", FIREBASE_FORECAST_PATH)

        # Keep the previous snapshot before writing the new one.
        if isinstance(old_next, dict) and old_next:
            _firebase_request("PUT", FIREBASE_PREVIOUS_DAY_PATH, old_next)
            old_date = str(old_next.get("forecast_date", "")).strip()
            if old_date:
                _firebase_request("PUT", f"{FIREBASE_HISTORY_BASE_PATH}/{old_date}", old_next)

        # Write the latest forecast for dashboard quick access.
        _firebase_request("PUT", FIREBASE_FORECAST_PATH, forecast_payload)

        # Also store current forecast into date-based history.
        current_date = str(forecast_payload.get("forecast_date", "")).strip()
        if current_date:
            _firebase_request("PUT", f"{FIREBASE_HISTORY_BASE_PATH}/{current_date}", forecast_payload)

        print("Forecast pushed to Firebase:")
        print(" - latest:", FIREBASE_FORECAST_PATH)
        print(" - previous:", FIREBASE_PREVIOUS_DAY_PATH)
        print(" - history:", f"{FIREBASE_HISTORY_BASE_PATH}/{current_date}")
    except Exception as exc:
        print(f"Push failed: {exc}")
        raise
else:
    print("Skip Firebase push. Fill FIREBASE_RTD_URL or set PUSH_FORECAST_TO_FIREBASE=True.")